# Imports de librerias

In [0]:
from pyspark.sql.functions import expr, lower, trim, col, length, datediff

# Lectura de la Tabla Bronce olist_orders

In [0]:
# creo el df con la tabla bronce de olist_customers
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_orders_dataset")

In [0]:
df.display()

In [0]:
print(df.count())

# Transformaciones

In [0]:
# cambio de tipo de dato
df = (
    df
    .withColumn("order_purchase_timestamp", expr("try_cast(order_purchase_timestamp as timestamp)"))
    .withColumn("order_approved_at", expr("try_cast(order_approved_at as timestamp)"))
    .withColumn("order_delivered_carrier_date", expr("try_cast(order_delivered_carrier_date as timestamp)"))
    .withColumn("order_delivered_customer_date", expr("try_cast(order_delivered_customer_date as timestamp)"))
    .withColumn("order_estimated_delivery_date", expr("try_cast(order_estimated_delivery_date as timestamp)"))
)


In [0]:
# paso a minuscula y saco espacios del order_status
df = df.withColumn(
    "order_status",
    trim(lower(col("order_status")))
)


In [0]:
# quito duplicados por order_id
df = df.dropDuplicates(["order_id"])

In [0]:
# validaciones
df = df.filter(
    (col("order_id").isNotNull()) &
    (length(col("order_id")) > 10) &
    (col("customer_id").isNotNull()) &
    (length(col("customer_id")) > 10) &
    (col("order_purchase_timestamp").isNotNull())
)

In [0]:

# tiempo de entrega
df = df.withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

In [0]:
print(df.count())

# Crear la tabla Silver de olist_orders

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_orders")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_orders